In [ ]:
import torch
import numpy as np
from typing import Optional, Tuple, List, Dict
from dataclasses import dataclass
import time
from scipy.sparse import diags
from pubo_to_qubo import PUBOToQUBOReducer


@dataclass
class VeloxQConfig:
    num_sweeps: int = 1000
    num_samples: int = 1024
    beta_start: float = 0.1
    beta_end: float = 5.0
    alpha_start: float = 0.3
    alpha_end: float = 1
    annealing_schedule: str = 'linear'
    auto_tune: bool = True
    custom_quality: bool = False
    custom_speed: bool = False
    quality_multiplier: float = 2.0
    speed_divider: float = 2.0
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    verbose: bool = False


class VeloxQSolver:
    
    def __init__(self, Q: np.ndarray, config: Optional[VeloxQConfig] = None):
        self.config = config or VeloxQConfig()
        self.device = torch.device(self.config.device)
        
        if self.device.type == 'cuda':
            torch.cuda.empty_cache()
        
        self.N = Q.shape[0]

        self.Q = torch.tensor(Q, dtype=torch.float32, device=self.device)

        self.Q_diag = self.Q.diag().clone()

        self._setup_parameters()
        
    
    def _setup_parameters(self):
        self.num_sweeps = self.config.num_sweeps
        self.num_samples = self.config.num_samples
        self.beta_start = self.config.beta_start
        self.beta_end = self.config.beta_end
        self.alpha_start = self.config.alpha_start
        self.alpha_end = self.config.alpha_end

    def _energy(self, x: torch.Tensor) -> torch.Tensor:
        return torch.einsum('bi,ij,bj->b', x, self.Q, x)
    
    def solve(self, x0=None) -> Tuple[np.ndarray, float, Dict]:
        
        if x0 is not None:
            x = x0
        else:
            x = torch.randint(0, 2, (self.num_samples, self.N), 
                             device=self.device).float()

        best_energy = float('inf')
        best_x = None
        energy_history = []
        data_x = []

        for sweep in range(self.num_sweeps):

            t = sweep / self.num_sweeps
            if self.config.annealing_schedule == 'linear':
                beta = self.beta_start + t * (self.beta_end - self.beta_start)
                alpha = self.alpha_start + t * (self.alpha_end - self.alpha_start)
            elif self.config.annealing_schedule == 'exp':
                beta = self.beta_start * (self.beta_end / self.beta_start) ** t
                alpha = self.alpha_start * (self.alpha_end / self.alpha_start) ** t
            else:
                print("annealing_schedule: 'linear' or 'exp'")
                return None

            
            field = self.Q_diag + 2 * (torch.matmul(x, self.Q) - self.Q_diag * x)

            delta_E = (1 - 2*x) * field

            accept_prob = torch.exp(-beta * delta_E)
            accept_prob = alpha * torch.clamp(accept_prob, max=1.0)
            flip = torch.rand_like(x) < accept_prob
            x = torch.where(flip, 1 - x, x)

            if sweep % 10 == 0:
                              
                data_x.append(x[:, :200].cpu() * 2 - 1)
                
                energies = self._energy(x)
                min_energy = energies.min().item()

                if min_energy < best_energy:
                    best_energy = min_energy
                    best_idx = energies.argmin().item()
                    best_x = x[best_idx].cpu().numpy().copy()

                if self.config.verbose:
                    print(f"  Sweep {sweep}, β={beta:.2f}, E={min_energy:.4f}")
        
        return best_x.astype(int), data_x
    
   

In [ ]:
def compute_energy_sparse(x, H1, H2, H3):

    energy = 0.0
    n = len(x)

    energy += np.dot(x, H1)

    for i in range(n):
        for j in range(i+1, n):
            if H2[i, j] != 0:
                energy += H2[i, j] * x[i] * x[j]

    for i in range(n):
        for j in range(i+1, n):
            for k in range(j+1, n):
                if H3[i, j, k] != 0:
                    energy += H3[i, j, k] * x[i] * x[j] * x[k]
    
    return energy


In [ ]:
for i in range(20, 50):
    reducer = PUBOToQUBOReducer()
    H1=np.zeros(200) 
    H2=np.zeros((200, 200)) 
    H3=np.load(f"J_T_h_1/T_{i}.npy")
    J, h, pair_to_aux = reducer.reduce(H1, H2, H3, penalty=np.max(abs(H3)))
    Q = ising_to_qubo_sparse(J, h)
    
    config1 = VeloxQConfig(
        num_sweeps=2500,
        num_samples=5000,
        auto_tune=False,
        verbose=False,
        device='cuda',
        annealing_schedule='exp',
        beta_start=0.05,
        beta_end=2.5,
        alpha_start=0.5,
        alpha_end=0.6,
    )

    solver1 = VeloxQSolver(Q.toarray()/np.std(Q.toarray()), config=config1)
    x1, data_x = solver1.solve()
    print(i, compute_energy_sparse((x1 * 2 - 1)[:200], H1, H2, H3))
    np.save(f"solution_data_1/Velox_q_{i}.npy", data_x)

In [ ]:
def ising_to_qubo(J, h):

    n = J.shape[0]
    Q = 4 * J.copy()
    np.fill_diagonal(Q, 0)
    linear = -2 * J.sum(axis=1) - 2 * J.sum(axis=0) + 2 * h
    np.fill_diagonal(Q, linear)
    
    return Q